In [ ]:
import os
import supervision as sv
from rfdetr import RFDETRLarge, RFDETRBase
from rfdetr.util.coco_classes import COCO_CLASSES

model = RFDETRBase()
# model_path = ""

def callback(frame, index):
    rgb_frame = frame[:, :, ::-1].copy()
    detections = model.predict(rgb_frame, threshold=0.5)
        
    labels = [
        f"{COCO_CLASSES[class_id]} {confidence:.2f}"
        for class_id, confidence
        in zip(detections.class_id, detections.confidence)
    ]

    annotated_frame = frame.copy()
    annotated_frame = sv.BoxAnnotator().annotate(annotated_frame, detections)
    annotated_frame = sv.LabelAnnotator().annotate(annotated_frame, detections, labels)
    return annotated_frame

import time
save_times = []
# base_path = "/home/ubuntu/retrain_pipeline/data/defense_high_res"
base_path = "/home/ubuntu/retrain_pipeline/data2/goals"
# dir_path = ["tackle_high_res", "defense_high_res", "goals_high_res", "chances_high_res"]
# for dire in dir_path:
    # base_path = os.path.join(base_path, dire)
# os.makedirs(os.path.join(base_path, "rf_base_annotated"), exist_ok=True)
# for vid in os.listdir(base_path):
#     start = time.time()
#     if vid.endswith(".mp4"):
#         print(f"Processing {vid}")
#         sv.process_video(
#             source_path=os.path.join(base_path, vid),
#             target_path=os.path.join(base_path, "rf_base_annotated", f"{vid[:-4]}_annotated.mp4"),
#             callback=callback
#         )
#         print(f"Processed {vid} in {time.time() - start:.2f} seconds")
#     save_times.append(time.time() -start)
# with open(os.path.join(base_path, "save_times_rf_detr_base.txt"), "w") as f:
#     for time in save_times:
#         f.write(f"{time}\n")


#For single video
# start = time.time()
# sv.process_video(
#     source_path='/home/ubuntu/devesh/GSIC/GSIC_cricket/data/2025-03-11_11-17-15.mp4',
#     target_path='/home/ubuntu/devesh/GSIC/GSIC_cricket/data/2025-03-11_11-17-15_annotated.mp4',
#     callback=callback
# )
# print(f"Totoal time taken is {time.time() - start}")


In [ ]:
#Load dataset
import supervision as sv

ds_train = sv.DetectionDataset.from_coco(
    images_directory_path=f'{dataset.location}/train',
    annotations_path=f'{dataset.location}/train/_annotations.coco.json',
)
ds_valid = sv.DetectionDataset.from_coco(
    images_directory_path=f'{dataset.location}/valid',
    annotations_path=f'{dataset.location}/valid/_annotations.coco.json',
)
ds_test = sv.DetectionDataset.from_coco(
    images_directory_path=f'{dataset.location}/test',
    annotations_path=f'{dataset.location}/test/_annotations.coco.json',
)

ds_train.classes
# ['person', 'bicycle', 'car', ...]

len(ds_train), len(ds_valid), len(ds_test)
# 800, 100, 100
import supervision as sv

ds = sv.DetectionDataset(...)

# Option 1
for image_path, image, annotations in ds:
    ... # Process each image and its annotations

# Option 2
# for idx in range(len(ds)):
#     image_path, image, annotations = ds[idx]
#     ... # Process the image and annotations at index `idx`

In [ ]:
import supervision as sv
from ultralytics import YOLO
model = YOLO('model')

test_set = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/test/images",
    annotations_directory_path=f"{dataset.location}/test/labels",
    data_yaml_path=f"{dataset.location}/data.yaml"
)

image_paths = []
predictions_list = []
targets_list = []

for image_path, image, label in test_set:
    result = model(image)[0]
    predictions = sv.Detections.from_ultralytics(result)

    image_paths.append(image_path)
    predictions_list.append(predictions)
    targets_list.append(label)

In [ ]:
# from supervision.metrics import MeanAveragePrecision, F1Score, MetricTarget
# map_metric = MeanAveragePrecision(metric_target = MetricTarget.MASKS)
# map_result = map_metric.update(predicions_list, targets_list).compute()
# f1_metric = F1Score(metric_target = MetricTarget.MASKS)
# f1_result = f1_metric.update(predictions_list, targets_list).compute()
#f1_result.plot()

In [37]:
import supervision as sv
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from rfdetr import RFDETRLarge, RFDETRBase
from rfdetr.util.coco_classes import COCO_CLASSES


RF_DETR= False
# model = RFDETRLarge()
YOLO_base_model = False
# model = YOLO('/home/ubuntu/retrain_pipeline/yolov8n.pt')
# model = YOLO('/home/ubuntu/retrain_pipeline/RF_DETR/yolo11l.pt')
YOLO_custom_model = True
model = YOLO('/Users/spectatr/Downloads/best_datav10_rectified_nano_1920_e261_final.pt')
# model = YOLO('/home/ubuntu/models/best_data_v7_200_refined.pt')

dataset = sv.DetectionDataset.from_coco(
    images_directory_path="/Users/spectatr/Downloads/nsl_data_test/coco_dataset/test", # images_directory_path="/home/ubuntu/retrain_pipeline/data/Data_v10/coco_dataset/valid2/valid",
    annotations_path="/Users/spectatr/Downloads/nsl_data_test/coco_dataset/_annotations.coco.json")

original_annotations = dataset.annotations.copy()
for image_name in dataset.annotations:
    detections = dataset.annotations[image_name]
    if len(detections) > 0:
        mask = detections.class_id == 0
        dataset.annotations[image_name] = detections[mask]
dataset.classes = [dataset.classes[0]]  # Keep only first class
print(len(dataset.images))
# def callback(image: np.ndarray) -> sv.Detections:
#     if RF_DETR:
#         rgb_frame = image[:, :, ::-1].copy()
#         detections = model.predict(rgb_frame, threshold = 0.5)
#         detections = detections[detections.class_id==37]
#         sorted_indices = np.argsort(detections.confidence)[::-1]
#         keep_indices = sorted_indices[:1]
#         detections = detections[keep_indices]
#         detections.class_id = np.zeros_like(detections.class_id)
#         # print(detections.class_id)
    
#     elif YOLO_base_model:
#         result = model(image, conf = 0.5)[0]
#         detections = sv.Detections.from_ultralytics(result)
#         if len(detections) > 0:
#             mask = detections.class_id == 32
#             detections = detections[mask]
#         keep_indices = np.argsort(detections.confidence)[::-1][:1] #Only top 1
#         detections = detections[keep_indices]
#         detections.class_id[:] = 0
    
#     elif YOLO_custom_model:
#         result = model(image, conf = 0.5)[0]
#         detections = sv.Detections.from_ultralytics(result)
#         keep_indices = np.argsort(detections.confidence)[::-1][:1] #Only top 1
#         detections = detections[keep_indices]
#         print(detections.class_id)
        
#     return detections

# confusion_matrix = sv.ConfusionMatrix.benchmark(dataset=dataset,callback=callback)  # or custom_callback)
# # image_names = list(dataset.images.keys()) # first_image = next(iter(dataset.images.values()))
# # image_name = image_names[18]
# # image_index = dataset.images[image_name]
# # predictions = callback(image_index)
# # annotated = sv.BoxAnnotator().annotate(image_index.copy(), predictions)
# # plt.imshow(annotated)
# print(confusion_matrix.matrix)

SupervisionWarnings: images is deprecated: `DetectionDataset.images` property is deprecated and will be removed in `supervision-0.26.0`. Iterate with `for path, image, annotation in dataset:` instead.


203


In [ ]:
# Collect all predictions and targets for other metrics
all_predictions = []
all_targets = []

for image_name, image in dataset.images.items():
    # Get predictions
    predictions = callback(image)
    all_predictions.append(predictions)
    
    # Get ground truth
    targets = dataset.annotations[image_name]
    all_targets.append(targets)

# Compute mAP
mean_ap_metric = sv.MeanAveragePrecision.from_detections(
    prediction_set=all_predictions,
    target_set=all_targets
)
map_result = mean_ap_metric.compute()

print(f"mAP@0.5: {map_result.map50:.3f}")
print(f"mAP@0.5:0.95: {map_result.map50_95:.3f}")